Project repository: [josepsuy/LPR-GNN](https://github.com/josepsuy/LPR-GNN) (currently private; owner-granted access required). See [README.md](README.md) for installation and [CITATION.cff](CITATION.cff) for the preferred scientific citation.

# Training graph generation

Generate labelled and normalized decision graphs from the reference schedules of training instances. The split manifest excludes validation and test instances. Existing graph files are preserved unless `OVERWRITE_EXISTING` is enabled.

Tied reference activities retain multiple positive targets, and subset exploration merges repeated subsets. States with a single Border activity are traversed but not saved as training graphs. Augmentation uses resource permutation and duration scaling.

Raw instances are not bundled. Before running this notebook, follow [DATASETS.md](DATASETS.md) for download sources, placement and frozen-split requirements.


## 1. Imports and configuration

In [ ]:
from pathlib import Path
from collections import Counter
from copy import deepcopy
from itertools import groupby, permutations
import json
import os
import random
import sys

from src.normalization import GraphNormalizer

import pandas as pd
import yaml
from IPython.display import display
from joblib import Parallel, delayed
from tqdm.auto import tqdm


PROJECT_ROOT = Path.cwd().resolve()
COMMONS_DIR = PROJECT_ROOT / "src"
ORIGIN_DIR = PROJECT_ROOT / "source_data"
ORIGIN_CSV = ORIGIN_DIR / "training_sources.csv"
CONFIG_PATH = PROJECT_ROOT / "config.yaml"

# Single output directory: data2 is not created.
OUTPUT_TRAIN_DIR = PROJECT_ROOT / "data" / "train"
RANDOM_SEED = int(yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))["seed"])
OVERWRITE_EXISTING = False
NORMALIZE_SOLUTION = False
RESOURCE_COUNT = 4
AUGMENT_WITH_RESOURCE_PERMUTATION = True
AUGMENT_WITH_DURATION_SCALING = True
DURATION_SCALE_MIN = 2
DURATION_SCALE_MAX = 5
VALIDATION_ERROR_EXAMPLES = 50

# Workers per instance, controlled by data.num_workers in config.yaml.
N_JOBS = int(yaml.safe_load(CONFIG_PATH.read_text(encoding='utf-8'))['data']['num_workers'])
PARALLEL_BACKEND = "loky"
PARALLEL_TEMP_DIR = PROJECT_ROOT / "outputs" / ".joblib_create_graphs"
PARALLEL_TEMP_DIR.mkdir(parents=True, exist_ok=True)

with CONFIG_PATH.open("r", encoding="utf-8") as stream:
    PROJECT_CONFIG = yaml.safe_load(stream) or {}

NORMALIZATION_CONFIG = PROJECT_CONFIG.get("normalization", {})
TEMPORAL_REFERENCE = float(NORMALIZATION_CONFIG.get("temporal_reference", 100.0))
RESOURCE_NORMALIZATION_METHOD = str(
    NORMALIZATION_CONFIG.get("resource_method", "instance_capacity_ratio")
)
if RESOURCE_NORMALIZATION_METHOD != "instance_capacity_ratio":
    raise ValueError("Resource normalization must be 'instance_capacity_ratio'.")

graph_normalizer = GraphNormalizer(temporal_reference=TEMPORAL_REFERENCE)

for name, path in {
    "COMMONS_DIR": COMMONS_DIR,
    "CONFIG_PATH": CONFIG_PATH,
    "ORIGIN_CSV": ORIGIN_CSV,
    "OUTPUT_TRAIN_DIR": OUTPUT_TRAIN_DIR,
}.items():
    print(f"{name:18s}: {path} | exists={path.exists()}")
print(f"N_JOBS            : {N_JOBS}")
print(f"PARALLEL_BACKEND  : {PARALLEL_BACKEND}")

## 2. Load project modules


In [ ]:
if not COMMONS_DIR.exists():
    raise FileNotFoundError(f"The src directory does not exist: {COMMONS_DIR}")

# rcpsp_schedule uses relative imports; import it as a member of the src package.
project_path = str(PROJECT_ROOT)
if project_path not in sys.path:
    sys.path.insert(0, project_path)

from src import rcpsp_instance
from src import rcpsp_schedule

print("rcpsp_instance loaded from:", rcpsp_instance.__file__)
print("rcpsp_schedule loaded from:", rcpsp_schedule.__file__)

## 3. Read and validate the source CSV

In [ ]:
if not ORIGIN_CSV.exists():
    raise FileNotFoundError(f"The CSV file does not exist: {ORIGIN_CSV}")

origin_df = pd.read_csv(ORIGIN_CSV, sep=";")
# Filter before graph generation; validation and test never enter training.
manifest = json.loads((PROJECT_ROOT / PROJECT_CONFIG['instance_evaluation']['manifest']).read_text(encoding='utf-8'))
allowed = {Path(e['path']).stem.lower() for e in manifest['splits']['train']}
# Resolve filename casing from the manifest for case-sensitive filesystems.
training_paths = {Path(e['path']).stem.lower(): Path(e['path']).relative_to('source_data').as_posix()
                  for e in manifest['splits']['train']}
origin_df = origin_df[origin_df['file'].astype(str).map(lambda value: Path(value).stem.lower()).isin(allowed)].copy()
if {Path(value).stem.lower() for value in origin_df['file'].astype(str)} != allowed:
    raise ValueError('Missing training instance labels in source data.')

required_columns = {"file", "sequence", "file_type"}
missing_columns = required_columns - set(origin_df.columns)
if missing_columns:
    raise ValueError(f"Missing CSV columns: {sorted(missing_columns)}")

output_stems = origin_df["file"].astype(str).map(lambda value: Path(value).stem)
duplicate_stems = output_stems[output_stems.duplicated(keep=False)]
if not duplicate_stems.empty:
    raise ValueError(
        "Duplicate output names: "
        + ", ".join(sorted(duplicate_stems.unique()))
    )

print(f"Instances: {len(origin_df)}")
display(origin_df.head())
origin_df['file'] = origin_df['file'].map(lambda value: training_paths[Path(value).stem.lower()])


## 4. Generate labelled decision graphs

The generator follows the reference schedule and explores proper subsets within tied activity groups. It labels eligible reference-consistent activities, adds typed original precedence relations, and yields graphs progressively without accumulating the dataset in memory.


In [ ]:
def parse_solution(raw_sequence):
    sequence = json.loads(raw_sequence) if isinstance(raw_sequence, str) else raw_sequence
    if not isinstance(sequence, list) or len(sequence) < 2:
        raise ValueError("The sequence must contain at least two elements.")

    solution = sequence[1:]
    if not isinstance(solution[-1], (list, tuple)) or len(solution[-1]) < 2:
        raise ValueError("The last element does not contain the expected makespan.")
    return solution


def edge_key_for(graph):
    return "edges" if "edges" in graph else "links"


def graph_edges(graph):
    return graph.get(edge_key_for(graph), [])


def build_solution_graph(
    nodes_dict,
    resource_levels,
    matrix,
    resource_time,
    border,
    others,
    original_matrix,
    positive_nodes,
):
    positive_nodes = set(positive_nodes) & set(border)
    if not positive_nodes:
        raise RuntimeError("The state has no possible target in Border.")

    all_nodes = sorted(border | others)
    base_graph = rcpsp_schedule.generate_dynamic_graph(
        nodes_dict,
        matrix,
        border,
        resource_time,
        all_nodes=all_nodes,
        positive_nodes=positive_nodes,
        resource_capacities=resource_levels,
    )
    return rcpsp_schedule.add_typed_original_relations(
        base_graph,
        original_matrix,
        nodes_dict,
        border,
        all_nodes,
    )


def generate_proper_subset_graphs(
    nodes_dict,
    resource_levels,
    matrix,
    resource_time,
    border,
    others,
    original_matrix,
    solution_nodes,
):
    """Yield one graph per reachable proper subset."""
    solution_nodes = tuple(dict.fromkeys(int(node) for node in solution_nodes))
    solution_set = frozenset(solution_nodes)
    if not solution_set:
        return

    expanded = set()

    def explore(
        selected,
        local_matrix,
        local_resource_time,
        local_border,
        local_others,
    ):
        state = frozenset(selected)
        if state in expanded:
            return
        expanded.add(state)

        remaining = solution_set - state
        if not remaining:
            return  # Do not generate the final state without possible targets.

        feasible_positives = set(remaining).intersection(local_border)
        if not feasible_positives:
            raise RuntimeError(
                "No remaining activity in the tied group is in Border: "
                f"remaining={sorted(remaining)}, border={sorted(local_border)}"
            )

        # Forced choices are traversed but never materialized or augmented.
        if len(local_border) > 1:
            yield build_solution_graph(
                nodes_dict, resource_levels, local_matrix, local_resource_time,
                local_border, local_others, original_matrix, feasible_positives,
            )

        for selected_node in sorted(feasible_positives):
            child_state = state | {selected_node}
            if child_state == solution_set or child_state in expanded:
                continue

            # update_schedule_state only modifies matrix and resource_time.
            child_matrix = [row.copy() for row in local_matrix]
            child_resource_time = local_resource_time.copy()
            child_border, child_others = rcpsp_schedule.update_schedule_state(
                nodes_dict,
                resource_levels,
                child_matrix,
                selected_node,
                local_border,
                local_others,
                child_resource_time,
            )
            yield from explore(
                selected + (selected_node,),
                child_matrix,
                child_resource_time,
                child_border,
                child_others,
            )

    yield from explore(
        tuple(),
        matrix,
        resource_time,
        set(border),
        set(others),
    )


def apply_solution_group(
    nodes_dict,
    resource_levels,
    matrix,
    resource_time,
    border,
    others,
    solution_nodes,
):
    """Apply the group to the main state so scheduling can continue."""
    pending = list(dict.fromkeys(int(node) for node in solution_nodes))
    while pending:
        selected_node = next((node for node in pending if node in border), None)
        if selected_node is None:
            raise RuntimeError(
                "Cannot continue the tied group: "
                f"pending={pending}, border={sorted(border)}"
            )
        border, others = rcpsp_schedule.update_schedule_state(
            nodes_dict,
            resource_levels,
            matrix,
            selected_node,
            border,
            others,
            resource_time,
        )
        pending.remove(selected_node)
    return border, others


def create_json_instances_from_solution(instance, max_time, solution):
    """Yield graphs progressively and merge repeated subsets."""
    original_matrix = rcpsp_instance.create_distance_matrix(instance)
    (
        nodes_dict,
        resource_levels,
        matrix,
        resource_time,
        _,
        border,
        others,
    ) = rcpsp_schedule.initialize_schedule_state(instance, max_time)

    ordered_solution = sorted(
        [list(item) for item in solution],
        key=lambda item: item[1],
    )
    decision_count = len(nodes_dict["nodes"]) - 3
    if len(ordered_solution) < decision_count:
        raise ValueError(
            f"The schedule has {len(ordered_solution)} decisions, but requires "
            f"at least {decision_count}."
        )

    decisions = ordered_solution[:decision_count]
    node_ids = [int(item[0]) for item in decisions]
    if len(node_ids) != len(set(node_ids)):
        raise ValueError("The schedule contains duplicate decision activities.")

    for _, tied_items in groupby(decisions, key=lambda item: item[1]):
        solution_nodes = [int(item[0]) for item in tied_items]
        yield from generate_proper_subset_graphs(
            nodes_dict,
            resource_levels,
            matrix,
            resource_time,
            border,
            others,
            original_matrix,
            solution_nodes,
        )
        border, others = apply_solution_group(
            nodes_dict,
            resource_levels,
            matrix,
            resource_time,
            border,
            others,
            solution_nodes,
        )


def select_duration_scale_factor(seed):
    return random.Random(int(seed)).randint(
        DURATION_SCALE_MIN,
        DURATION_SCALE_MAX,
    )


def generate_base_graphs(row, position):
    random.seed(RANDOM_SEED + int(position))
    source_path = ORIGIN_DIR / str(row["file"])
    solution = parse_solution(row["sequence"])
    max_time = int(solution[-1][1]) * 2

    instance = rcpsp_instance.parse_mrcpsp(
        source_path,
        row["file_type"],
    )
    duration_scale_factor = select_duration_scale_factor(
        RANDOM_SEED + 1_000_000 + int(position)
    )
    graphs = create_json_instances_from_solution(instance, max_time, solution)
    return source_path, graphs, duration_scale_factor

## 5. Augmentation and normalization

Each base graph retains its original variant. Optional augmentation creates a resource-permuted copy and a duration-scaled copy with an integer factor from 2 to 5; these transformations are not combined. Temporal scaling rebuilds the bidirectional Time-Resources relations with scales `[1, 8, 64, 512]`.

The shared normalizer applies `log1p` with a fixed temporal reference and no clipping to durations and deltas. Resource features are divided by instance capacity, and Other-to-Border proximity is normalized locally for each Border node.


In [ ]:
def select_resource_permutation(seed, resource_count=RESOURCE_COUNT):
    identity = tuple(range(resource_count))
    candidates = [
        permutation
        for permutation in permutations(range(resource_count))
        if permutation != identity
    ]
    if not candidates:
        raise ValueError("At least two resources are required for permutation.")
    return random.Random(int(seed)).choice(candidates)


def permute_resources(graph, permutation):
    """Consistently permute all resource vectors in a graph."""
    permutation = tuple(int(index) for index in permutation)
    resource_count = len(permutation)
    if sorted(permutation) != list(range(resource_count)):
        raise ValueError(f"Invalid resource permutation: {permutation}")

    output = deepcopy(graph)

    def visit(value):
        if isinstance(value, dict):
            for field, child in list(value.items()):
                field_lower = str(field).lower()
                is_resource_field = (
                    "resource" in field_lower
                )
                is_resource_vector = (
                    isinstance(child, list)
                    and len(child) == resource_count
                    and all(isinstance(item, (int, float)) for item in child)
                )
                if is_resource_field and is_resource_vector:
                    value[field] = [child[index] for index in permutation]
                else:
                    visit(child)
        elif isinstance(value, list):
            for child in value:
                visit(child)

    visit(output)
    return output


# One implementation shared with inference.
# - resources: quantity / instance capacity, [0,1]
# - durations and deltas: log1p(x) / log1p(TEMPORAL_REFERENCE), without clipping
# - Other -> Border proximity: inverse raw delta, normalized per Border

TIME_NODE_PREFIX = "time-resources"


def time_node_index(node_id):
    text = str(node_id)
    if not text.startswith(TIME_NODE_PREFIX):
        return None
    suffix = text[len(TIME_NODE_PREFIX):]
    try:
        return int(suffix)
    except ValueError as exc:
        raise ValueError(f"Invalid time-node identifier: {node_id}") from exc


def scale_temporal_fields(record, factor):
    output = deepcopy(record)
    for field, value in list(output.items()):
        if field.startswith("duration") or field.startswith("delta_"):
            if not isinstance(value, (int, float)):
                raise TypeError(f"Temporal field {field} is not numeric: {value!r}")
            output[field] = value * factor
    return output


def scale_temporal_graph(graph, factor):
    """Derive the scaled graph without rerunning scheduling."""
    factor = int(factor)
    if factor < 1:
        raise ValueError("The temporal scale factor must be positive.")

    edge_key = edge_key_for(graph)
    output = {
        field: deepcopy(value)
        for field, value in graph.items()
        if field not in {"nodes", "edges", "links"}
    }

    activity_nodes = []
    time_nodes = {}
    for raw_node in graph.get("nodes", []):
        index = time_node_index(raw_node.get("id"))
        if index is None:
            activity_nodes.append(scale_temporal_fields(raw_node, factor))
        else:
            if index in time_nodes:
                raise ValueError(f"Duplicate time node: {index}")
            time_nodes[index] = raw_node

    if not time_nodes:
        raise ValueError("The graph has no Time-Resources nodes.")
    original_times = sorted(time_nodes)
    expected_times = list(range(original_times[0], original_times[-1] + 1))
    if original_times != expected_times:
        raise ValueError("Time-Resources nodes do not form a consecutive sequence.")

    scaled_time_nodes = []
    scaled_times = []
    terminal_time = original_times[-1]
    scaled_minimum_time = original_times[0] * factor
    scaled_terminal_time = terminal_time * factor
    scaled_temporal_span = scaled_terminal_time - scaled_minimum_time
    for time_index in original_times:
        repeats = factor if time_index < terminal_time else 1
        for offset in range(repeats):
            scaled_time = time_index * factor + offset
            node = deepcopy(time_nodes[time_index])
            node["id"] = f"{TIME_NODE_PREFIX}{scaled_time}"
            node["relative_time"] = (
                0.0
                if scaled_temporal_span == 0
                else (scaled_time - scaled_minimum_time) / scaled_temporal_span
            )
            scaled_time_nodes.append(node)
            scaled_times.append(scaled_time)
    output["nodes"] = activity_nodes + scaled_time_nodes

    def scaled_endpoint(endpoint):
        index = time_node_index(endpoint)
        if index is None:
            return deepcopy(endpoint)
        return f"{TIME_NODE_PREFIX}{index * factor}"

    scaled_edges = []
    temporal_edges_inserted = False

    def append_temporal_multiscale_edges():
        generated_edges = rcpsp_schedule.build_temporal_multiscale_edges(
            scaled_times[0],
            scaled_times[-1] + 1,
        )
        for edge in generated_edges:
            edge["key"] = 0
            scaled_edges.append(edge)

    for raw_edge in graph.get(edge_key, []):
        if "way_time" in raw_edge:
            if not temporal_edges_inserted:
                append_temporal_multiscale_edges()
                temporal_edges_inserted = True
            continue

        edge = scale_temporal_fields(raw_edge, factor)
        edge["source"] = scaled_endpoint(edge.get("source"))
        edge["target"] = scaled_endpoint(edge.get("target"))
        scaled_edges.append(edge)

    if len(scaled_times) > 1 and not temporal_edges_inserted:
        raise ValueError("Missing Time-Resources edges.")

    output[edge_key] = scaled_edges
    return output


def tag_augmentation_variant(graph, variant):
    """Tag the variant so loaders do not depend on physical ordering."""
    graph.setdefault("graph", {})["augmentation_variant"] = variant
    return graph


def iter_augmented_graph_variants(graph, permutation, duration_scale_factor):
    """Yield the three tagged variants in order without accumulating them."""
    yield "original", tag_augmentation_variant(graph, "original")
    if AUGMENT_WITH_RESOURCE_PERMUTATION:
        yield "resource_permuted", tag_augmentation_variant(
            permute_resources(graph, permutation),
            "resource_permuted",
        )
    if AUGMENT_WITH_DURATION_SCALING:
        yield "duration_scaled", tag_augmentation_variant(
            scale_temporal_graph(graph, duration_scale_factor),
            "duration_scaled",
        )


def normalize_graph_shared(graph):
    return graph_normalizer(graph)

## 6. Validate labels and original relations

In [ ]:
ORIGINAL_RELATION_INVERSE = {
    "original_border_other_forward": "original_border_other_reverse",
    "original_border_other_reverse": "original_border_other_forward",
    "original_other_other_forward": "original_other_other_reverse",
    "original_other_other_reverse": "original_other_other_forward",
}
ORIGINAL_RELATION_ENDPOINTS = {
    "original_border_other_forward": ("border", "other"),
    "original_border_other_reverse": ("other", "border"),
    "original_other_other_forward": ("other", "other"),
    "original_other_other_reverse": ("other", "other"),
}
ORIGINAL_RELATION_WAY = {
    "original_border_other_forward": 1,
    "original_border_other_reverse": 0,
    "original_other_other_forward": 1,
    "original_other_other_reverse": 0,
}


def validate_graph(graph, source_file, graph_idx, variant):
    nodes = graph.get("nodes", [])
    edges = graph_edges(graph)
    entity_by_id = {str(node.get("id")): node.get("entity") for node in nodes}

    border_nodes = [node for node in nodes if node.get("entity") == "border"]
    positives = sum(node.get("solution") in (1, 1.0, True, "1") for node in border_nodes)

    original_edges = [
        edge for edge in edges
        if edge.get("relation") in ORIGINAL_RELATION_INVERSE
    ]
    original_index = {
        (str(edge.get("source")), str(edge.get("target")), edge.get("relation")): edge
        for edge in original_edges
    }
    original_errors = []

    metadata = graph.get("graph", {})
    capacities = metadata.get("resource_capacities") if isinstance(metadata, dict) else None
    resource_values = []
    for node in nodes:
        for field, value in node.items():
            if isinstance(value, list) and "resource" in field.lower():
                resource_values.extend(float(item) for item in value)
    for edge in edges:
        for field, value in edge.items():
            if isinstance(value, list) and "resource" in field.lower():
                resource_values.extend(float(item) for item in value)
    resource_ratios_valid = bool(capacities) and all(0.0 <= value <= 1.0 for value in resource_values)

    for edge in original_edges:
        source = str(edge.get("source"))
        target = str(edge.get("target"))
        relation = edge.get("relation")
        way = edge.get("way_original_edges")
        inverse_relation = ORIGINAL_RELATION_INVERSE[relation]
        inverse = original_index.get((target, source, inverse_relation))
        endpoints = (entity_by_id.get(source), entity_by_id.get(target))
        separate = not any(
            key.startswith("way_") and key != "way_original_edges"
            for key in edge
        )

        valid = (
            endpoints == ORIGINAL_RELATION_ENDPOINTS[relation]
            and way == ORIGINAL_RELATION_WAY[relation]
            and separate
            and inverse is not None
            and inverse.get("way_original_edges") == 1 - way
            and inverse.get("delta_original_edges") == edge.get("delta_original_edges")
        )
        if not valid:
            original_errors.append({
                "source": source,
                "target": target,
                "relation": relation,
                "endpoints": endpoints,
            })

    return {
        "source_file": str(source_file),
        "graph_idx": graph_idx,
        "variant": variant,
        "n_border": len(border_nodes),
        "n_solution_1": positives,
        "labels_valid": positives >= 1,
        "multigraph": graph.get("multigraph") is True,
        "original_edges": len(original_edges),
        "original_edges_valid": bool(original_edges) and not original_errors,
        "resource_ratios_valid": resource_ratios_valid,
        "original_errors": original_errors,
    }


def aggregate_validation(result, counters, distribution, error_examples):
    """Keep global counters and only a bounded error sample."""
    counters["graphs"] += 1
    counters["border_nodes_total"] += int(result["n_border"])
    counters["positive_labels_total"] += int(result["n_solution_1"])
    counters["original_edges_total"] += int(result["original_edges"])
    distribution[int(result["n_solution_1"])] += 1

    checks = (
        ("labels_valid", "labels_valid", "labels_invalid"),
        ("multigraph", "multigraph_valid", "multigraph_invalid"),
        ("original_edges_valid", "original_edges_valid", "original_edges_invalid"),
        ("resource_ratios_valid", "resource_ratios_valid", "resource_ratios_invalid"),
    )
    invalid = False
    for field, valid_counter, invalid_counter in checks:
        if result[field]:
            counters[valid_counter] += 1
        else:
            counters[invalid_counter] += 1
            invalid = True

    if invalid and len(error_examples) < VALIDATION_ERROR_EXAMPLES:
        error_examples.append(result)

## 7. Generate and save training graphs

Each worker processes an independent instance and writes its graphs to a dedicated temporary JSON file. Variants are normalized, validated and written progressively. After successful completion, the temporary file atomically replaces the final file in `data/train`. The main process aggregates generation status and validation counters.


In [ ]:
OUTPUT_TRAIN_DIR.mkdir(parents=True, exist_ok=True)


def process_origin_row(position, row):
    """Generate a complete instance within a single worker process."""
    output_path = OUTPUT_TRAIN_DIR / f"{Path(str(row['file'])).stem}.json"
    temporary_path = output_path.with_suffix(output_path.suffix + ".tmp")

    if output_path.exists() and not OVERWRITE_EXISTING:
        return {
            "position": position,
            "generation": {
                "position": position,
                "status": "SKIPPED",
                "output": str(output_path),
                "base_graphs": 0,
                "graphs": 0,
                "resource_permutation": "",
                "duration_scale_factor": "",
                "error": "Already exists",
            },
            "validation": {},
            "distribution": {},
            "error_examples": [],
        }

    local_validation = Counter()
    local_distribution = Counter()
    local_error_examples = []
    base_graph_count = 0
    graph_count = 0

    try:
        _, base_graphs, duration_scale_factor = generate_base_graphs(
            row,
            position,
        )

        resource_permutation = select_resource_permutation(
            RANDOM_SEED + int(position),
            resource_count=RESOURCE_COUNT,
        )

        with temporary_path.open("w", encoding="utf-8") as output_file:
            output_file.write("[")
            first_graph = True

            for raw_graph in base_graphs:
                base_graph_count += 1
                for variant, augmented_graph in iter_augmented_graph_variants(
                    raw_graph,
                    resource_permutation,
                    duration_scale_factor,
                ):
                    normalized_graph = normalize_graph_shared(augmented_graph)
                    validation_result = validate_graph(
                        normalized_graph,
                        output_path,
                        graph_count,
                        variant,
                    )
                    aggregate_validation(
                        validation_result,
                        local_validation,
                        local_distribution,
                        local_error_examples,
                    )

                    if not first_graph:
                        output_file.write(",")
                    json.dump(
                        normalized_graph,
                        output_file,
                        ensure_ascii=False,
                        separators=(",", ":"),
                    )
                    first_graph = False
                    graph_count += 1

            output_file.write("]")

        # A fully forced instance validly produces [] (no learning decisions).

        temporary_path.replace(output_path)
        generation = {
            "position": position,
            "status": "OK",
            "output": str(output_path),
            "base_graphs": base_graph_count,
            "graphs": graph_count,
            "resource_permutation": str(resource_permutation),
            "duration_scale_factor": duration_scale_factor,
            "error": "",
        }
        return {
            "position": position,
            "generation": generation,
            "validation": dict(local_validation),
            "distribution": dict(local_distribution),
            "error_examples": local_error_examples,
        }
    except Exception as exc:
        temporary_path.unlink(missing_ok=True)
        return {
            "position": position,
            "generation": {
                "position": position,
                "status": "ERROR",
                "output": str(output_path),
                "base_graphs": base_graph_count,
                "graphs": 0,
                "resource_permutation": "",
                "duration_scale_factor": "",
                "error": str(exc),
            },
            "validation": {},
            "distribution": {},
            "error_examples": [],
        }


generation_rows = []
validation_counters = Counter()
solution_distribution_counts = Counter()
validation_error_examples = []

task_iterator = (
    delayed(process_origin_row)(
        position,
        {"file": file_name, "sequence": sequence, "file_type": file_type},
    )
    for position, (file_name, sequence, file_type) in enumerate(
        origin_df[["file", "sequence", "file_type"]].itertuples(
            index=False,
            name=None,
        ),
        start=1,
    )
)

parallel_results = Parallel(
    n_jobs=N_JOBS,
    backend=PARALLEL_BACKEND,
    batch_size=1,
    pre_dispatch=N_JOBS,
    return_as="generator_unordered",
    temp_folder=str(PARALLEL_TEMP_DIR),
)(task_iterator)

progress = tqdm(total=len(origin_df), desc="Generating graphs", unit="source_file")
for completed in parallel_results:
    generation_rows.append(completed["generation"])
    validation_counters.update(completed["validation"])
    solution_distribution_counts.update(completed["distribution"])
    remaining_error_slots = max(
        0,
        VALIDATION_ERROR_EXAMPLES - len(validation_error_examples),
    )
    validation_error_examples.extend(
        completed["error_examples"][:remaining_error_slots]
    )
    progress.update(1)
progress.close()

generation_rows.sort(key=lambda item: item["position"])
generation_df = pd.DataFrame(
    generation_rows,
    columns=[
        "status", "output", "base_graphs", "graphs",
        "resource_permutation", "duration_scale_factor", "error",
    ],
)

display(generation_df["status"].value_counts().rename_axis("status").to_frame("files"))
display(generation_df[generation_df["status"] == "ERROR"])

## 8. Final validation summary

In [ ]:
validation_summary_df = pd.DataFrame([{
    "graphs": int(validation_counters["graphs"]),
    "labels_invalid": int(validation_counters["labels_invalid"]),
    "multigraph_invalid": int(validation_counters["multigraph_invalid"]),
    "original_edges_invalid": int(validation_counters["original_edges_invalid"]),
    "resource_ratios_invalid": int(validation_counters["resource_ratios_invalid"]),
    "original_edges_total": int(validation_counters["original_edges_total"]),
    "border_nodes_total": int(validation_counters["border_nodes_total"]),
    "positive_labels_total": int(validation_counters["positive_labels_total"]),
    "error_examples_stored": len(validation_error_examples),
}])
validation_errors_df = pd.DataFrame(validation_error_examples)

display(validation_summary_df.T)
display(validation_errors_df)
# Do not proceed to training with incomplete or invalid generated data.
if (generation_df['status'] == 'ERROR').any():
    raise RuntimeError('Graph generation failed for one or more instances; inspect generation_df.')
invalid_keys = ('labels_invalid', 'multigraph_invalid', 'original_edges_invalid', 'resource_ratios_invalid')
if any(validation_counters[key] for key in invalid_keys):
    raise RuntimeError('Generated graph validation failed; inspect validation_summary_df.')


## 9. Label distribution

The table reports positive targets per graph. A valid graph has at least one positive target. For tied activities, subset generation creates unique states with the remaining possible targets; the final zero-positive state is not written.


In [ ]:
total_graphs = sum(solution_distribution_counts.values())
solution_distribution_df = pd.DataFrame([
    {
        "n_solution_1": n_solution_1,
        "n_graphs": n_graphs,
        "percent": 100 * n_graphs / total_graphs,
    }
    for n_solution_1, n_graphs in sorted(solution_distribution_counts.items())
])
if solution_distribution_df.empty:
    solution_distribution_df = pd.DataFrame(
        columns=["n_solution_1", "n_graphs", "percent"]
    )

display(solution_distribution_df)